# Setup

In [2]:
import copy
import pandas as pd
import numpy as np
import time
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, average_precision_score, brier_score_loss
from sklearn.neighbors import KNeighborsClassifier
from sktime.transformations.panel.catch22 import Catch22
from scipy.special import expit
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler

In [3]:
# Random state seed
RAND_STATE = 4

In [4]:
# Load the normalized data
output_dir = Path('../processed/')

data = np.load(output_dir / 'normalized_split.npz')
X_train_norm = data['X_train']
X_val_norm = data['X_val']
X_test_norm = data['X_test']
y_train = data['y_train']
y_val = data['y_val']
y_test = data['y_test']

with open(output_dir / 'norm_params.pkl', 'rb') as f_in:
    norm_params = pickle.load(f_in)

print("Pre-processed data loaded successfully")

Pre-processed data loaded successfully


In [5]:
for name, X, y in [("TRAINING", X_train_norm, y_train),
                   ("VALIDATION", X_val_norm, y_val),
                   ("TEST", X_test_norm, y_test)]:
    n_sep = int((y == 1).sum())
    n_nsep = int((y == 0).sum())

    print("================")
    print(f"{name} DATA")
    print("================")
    print(f"X shape:    {X.shape}")
    print(f"y shape:    {y.shape}")
    print(f"NSEP count: {n_nsep}")
    print(f"SEP count:  {n_sep}")
    print(f"Ratio:      {n_nsep / n_sep:.2f}:1")
    print()

TRAINING DATA
X shape:    (12455, 288, 10)
y shape:    (12455,)
NSEP count: 12337
SEP count:  118
Ratio:      104.55:1

VALIDATION DATA
X shape:    (1780, 288, 10)
y shape:    (1780,)
NSEP count: 1763
SEP count:  17
Ratio:      103.71:1

TEST DATA
X shape:    (3559, 288, 10)
y shape:    (3559,)
NSEP count: 3525
SEP count:  34
Ratio:      103.68:1



In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)
torch.backends.cudnn.benchmark = True 

X_train_t = torch.as_tensor(X_train_norm, dtype=torch.float32, device=device)
X_val_t = torch.as_tensor(X_val_norm,   dtype=torch.float32, device=device)
X_test_t = torch.as_tensor(X_test_norm,  dtype=torch.float32, device=device)

y_train_t = torch.as_tensor(y_train, dtype=torch.long, device=device)
y_val_t = torch.as_tensor(y_val, dtype=torch.long, device=device)
y_test_t = torch.as_tensor(y_test, dtype=torch.long, device=device)


device: mps


# Evaluation Metrics

In [7]:
def true_skill_statistic(confusion_matrix):
    tn, fp, fn, tp = confusion_matrix.ravel()

    # Guard against divide by zero
    if (tp + fn == 0 or
        tn + fp == 0):
        return 0.0

    tss = tp / (tp + fn) - fp / (tn + fp)
    
    return float(tss)

def heidke_skill_score(confusion_matrix):
    tn, fp, fn, tp = confusion_matrix.ravel()

    numerator = 2 * ((tp * tn) - (fp * fn))
    denominator = ((tp + fn) * (fn + tn)) + ((tn + fp) * (tp + fp))

    # Guard against divide by zero
    if denominator == 0:
        return 0.0
        
    return float(numerator / denominator)

def brier_skill_score(y_true, y_prob):
    base_rate = np.mean(y_true)
    brier_ref = brier_score_loss(y_true, np.full(len(y_true), base_rate))
    return 1 - brier_score_loss(y_true, y_prob) / brier_ref

def evaluate_model_metrics( 
        y_actual, 
        y_pred, 
        y_prob, 
        model_name="Model"
    ):    
    conf_matrix = confusion_matrix(y_actual, y_pred)

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_actual, y_pred),
        "Precision": precision_score(y_actual, y_pred, zero_division=0),
        "Recall": recall_score(y_actual, y_pred, zero_division=0),
        "F1-Score": f1_score(y_actual, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_actual, y_prob),
        "PR-AUC": average_precision_score(y_actual, y_prob),
        "Brier": brier_score_loss(y_actual, y_prob),
        "BSS" : brier_skill_score(y_actual, y_prob),
        "TSS": true_skill_statistic(conf_matrix),
        "HSS2": heidke_skill_score(conf_matrix),
        "Confusion Matrix": conf_matrix
    }
    
    return metrics

def print_metrics(metrics):
    print("===================")
    print("PERFORMANCE METRICS")
    print("===================")
    for name, val in metrics.items():
        formatted_val = round(val,5) if isinstance(val, (float, np.floating)) else val   
        print(f"{name:<17}: {formatted_val}")
    print()

# InfoNCE

In [ ]:
# ---- Model (outputs L2-normalized embeddings) ----
class Encoder(nn.Module):
    def __init__(self, n_features, embed_dim=32, hidden=64):
        super().__init__()
        self.gru = nn.GRU(n_features, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, embed_dim)

    def forward(self, x):
        _, h = self.gru(x)
        return F.normalize(self.fc(h[-1]), dim=1)

# ---- Augmentation: jitter + per-channel scaling ----
def augment(x):
    noise = 0.1 * torch.randn_like(x)
    scale = 1 + 0.1 * torch.randn(x.shape[0], 1, x.shape[2], device=x.device)
    return x * scale + noise

# ---- Supervised contrastive loss ----
def supcon_loss(z, y, temp=0.1):
    sim = z @ z.T / temp                                   # (2B, 2B)
    eye = torch.eye(len(y), dtype=torch.bool, device=z.device)
    sim = sim.masked_fill(eye, -1e9)                       # ignore self-similarity
    log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True)
    pos = (y[:, None] == y[None, :]) & ~eye                # same label, not self
    return -(log_prob.masked_fill(~pos, 0).sum(1) / pos.sum(1)).mean()

# ---- Evaluation helpers ----
def embed(model, X, batch_size=512):
    model.eval()
    with torch.no_grad():
        out = [model(X[i:i + batch_size]).cpu() for i in range(0, len(X), batch_size)]
    return torch.cat(out).numpy()

def tss(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return tp / (tp + fn) + tn / (tn + fp) - 1

def predict_proba(model, X_ref, y_ref, X_eval, k=50):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(embed(model, X_ref), y_ref.numpy())            # reference = real (unbalanced) training data
    return knn.predict_proba(embed(model, X_eval))[:, 1]

thresholds = np.linspace(0.01, 0.99, 99)

def best_threshold(y_true, p):
    scores = [tss(y_true, (p >= t).astype(int)) for t in thresholds]
    i = int(np.argmax(scores))
    return thresholds[i], scores[i]

# ---- Balanced batches (roughly 50/50) ----
class_counts = torch.bincount(y_train_t)
weights = (1.0 / class_counts.float())[y_train_t]
sampler = WeightedRandomSampler(weights, num_samples=len(y_train_t), replacement=True)
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=256, sampler=sampler)

# ---- Training ----
model = Encoder(n_features=X_train_t.shape[-1])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

best_val_tss, best_state, best_thr = -1.0, None, 0.5

for epoch in range(20):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        z = model(torch.cat([augment(xb), augment(xb)]))   # two views -> (2B, D)
        loss = supcon_loss(z, torch.cat([yb, yb]))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    p_val = predict_proba(model, X_train_t, y_train_t, X_val_t)
    val_auc = roc_auc_score(y_val_t.numpy(), p_val)
    thr, val_tss = best_threshold(y_val_t.numpy(), p_val)

    if val_tss > best_val_tss:
        best_val_tss, best_state, best_thr = val_tss, copy.deepcopy(model.state_dict()), thr

    print(f"epoch {epoch}: loss {train_loss:.4f} | val AUC {val_auc:.3f} | "
          f"val TSS {val_tss:.4f} @ thr {thr:.2f}")

# ---- Final test with best checkpoint and validation-chosen threshold ----
model.load_state_dict(best_state)
p_test = predict_proba(model, X_train_t, y_train_t, X_test_t)
test_tss = tss(y_test_t.numpy(), (p_test >= best_thr).astype(int))
test_auc = roc_auc_score(y_test_t.numpy(), p_test)
print(f"best val TSS {best_val_tss:.4f} (thr {best_thr:.2f}) | test TSS {test_tss:.4f} | test AUC {test_auc:.3f}")

epoch 0: loss 6.4061 | val AUC 0.742 | val TSS 0.4189 @ thr 0.01
epoch 1: loss 6.1559 | val AUC 0.782 | val TSS 0.4607 @ thr 0.01
epoch 2: loss 6.1322 | val AUC 0.767 | val TSS 0.4737 @ thr 0.01
epoch 3: loss 6.1208 | val AUC 0.733 | val TSS 0.4629 @ thr 0.03
epoch 4: loss 6.1089 | val AUC 0.704 | val TSS 0.4160 @ thr 0.03
epoch 5: loss 6.0976 | val AUC 0.748 | val TSS 0.4223 @ thr 0.01
epoch 6: loss 6.0846 | val AUC 0.728 | val TSS 0.4126 @ thr 0.03
epoch 7: loss 6.0510 | val AUC 0.670 | val TSS 0.2996 @ thr 0.09
epoch 8: loss 6.0208 | val AUC 0.669 | val TSS 0.3375 @ thr 0.06
epoch 9: loss 5.9968 | val AUC 0.693 | val TSS 0.4127 @ thr 0.05
epoch 10: loss 5.9574 | val AUC 0.623 | val TSS 0.3093 @ thr 0.06
epoch 11: loss 5.9246 | val AUC 0.649 | val TSS 0.2687 @ thr 0.01
epoch 12: loss 5.8803 | val AUC 0.561 | val TSS 0.1139 @ thr 0.01
epoch 13: loss 5.8336 | val AUC 0.600 | val TSS 0.1943 @ thr 0.01
epoch 14: loss 5.7784 | val AUC 0.549 | val TSS 0.0948 @ thr 0.01
epoch 15: loss 5.747